## Baseline

## Baseline: CatBoost для рекомендации фильмов

**Цель:** построить baseline-модель ранжирования фильмов на основе истории взаимодействий пользователей с контентом.

### Данные
Используются взаимодействия пользователей с фильмами и сериалами, пользовательские и контентные признаки, а также признаки истории просмотров, рассчитанные на временной срез.

### Метод
В качестве основной модели используется CatBoostClassifier с функцией потерь Logloss и подбором числа итераций по валидационной выборке с early stopping. Модель получает пары «пользователь — фильм» и предсказывает вероятность положительного взаимодействия.

Для сравнения используется простой baseline популярности, ранжирующий фильмы по числу взаимодействий за последние 30 дней.

### Оценка
Качество классификации оценивается с помощью ROC-AUC, Average Precision и LogLoss. Качество рекомендаций оценивается с помощью NDCG@10, Recall@10 и Precision@10 на кандидатах каждого пользователя и временного среза.

### Ограничения
Отсутствие взаимодействия не означает, что пользователь не заинтересован в фильме. Поэтому отрицательные примеры представляют собой непросмотренные кандидаты, а не подтверждённые негативные оценки. Метрики ранжирования рассчитываются на сформированном наборе кандидатов и не эквивалентны оценке по полному каталогу.

Baseline служит точкой отсчёта для дальнейшего эксперимента с автоматическим построением признаков.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
)

SEED = 42
TOP_K = 10

DATA_DIR = Path("data_processed_100neg")
if not DATA_DIR.exists():
    DATA_DIR = Path("data_processed")

OUTPUT_DIR = Path("results_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


Data directory: D:\tlab-research-1\data_processed_100neg
Output directory: D:\tlab-research-1\results_baseline


In [2]:
def load_split(name):
    path = DATA_DIR / f"{name}_features.parquet"

    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")

    df = pd.read_parquet(path)
    df["cutoff_date"] = pd.to_datetime(df["cutoff_date"])
    df["label"] = df["label"].astype(int)

    print(
        f"{name}: rows={len(df):,}, "
        f"users={df['user_id'].nunique():,}, "
        f"items={df['item_id'].nunique():,}, "
        f"positive_rate={df['label'].mean():.4f}"
    )

    return df


train = load_split("train")
valid = load_split("valid")
test = load_split("test")

required_columns = {
    "user_id", "item_id", "cutoff_date", "label"
}

for name, df in [
    ("train", train),
    ("valid", valid),
    ("test", test),
]:
    missing = required_columns - set(df.columns)
    assert not missing, f"{name}: отсутствуют колонки {missing}"

print("\nВсе выборки загружены.")

train: rows=1,799,113, users=7,772, items=15,963, positive_rate=0.0226
valid: rows=169,207, users=1,650, items=15,962, positive_rate=0.0249
test: rows=354,870, users=2,777, items=15,963, positive_rate=0.0247

Все выборки загружены.


In [3]:
excluded_columns = {"label", "cutoff_date"}

feature_columns = [
    col for col in train.columns
    if col not in excluded_columns
    and col in valid.columns
    and col in test.columns
]

# Категориальные признаки
categorical_features = [
    col for col in feature_columns
    if (
        col in {"user_id", "item_id"}
        or train[col].dtype == "object"
        or isinstance(train[col].dtype, pd.CategoricalDtype)
    )
]

def prepare_features(df, columns, categorical_columns):
    X = df[columns].copy()

    for col in columns:
        if col in categorical_columns:
            X[col] = (
                X[col]
                .fillna("__MISSING__")
                .astype(str)
            )
        else:
            X[col] = pd.to_numeric(
                X[col], errors="coerce"
            )
            X[col] = X[col].replace(
                [np.inf, -np.inf], np.nan
            ).fillna(0)

    return X


X_train = prepare_features(
    train, feature_columns, categorical_features
)
X_valid = prepare_features(
    valid, feature_columns, categorical_features
)
X_test = prepare_features(
    test, feature_columns, categorical_features
)

y_train = train["label"]
y_valid = valid["label"]
y_test = test["label"]

print("Количество признаков:", len(feature_columns))
print("Категориальные признаки:", categorical_features)

Количество признаков: 25
Категориальные признаки: ['user_id', 'item_id']


In [4]:
model = CatBoostClassifier(
    iterations=1200,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    early_stopping_rounds=100,
    allow_writing_files=False,
    verbose=200,
)

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
)

print("Best iteration:", model.best_iteration_)

0:	test: 0.9006947	best: 0.9006947 (0)	total: 540ms	remaining: 10m 48s
200:	test: 0.9298075	best: 0.9298075 (200)	total: 1m 39s	remaining: 8m 16s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9301927257
bestIteration = 264

Shrink model to first 265 iterations.
Best iteration: 264


In [5]:
valid_pred = model.predict_proba(X_valid)[:, 1]
test_pred = model.predict_proba(X_test)[:, 1]

classification_results = pd.DataFrame([
    {
        "split": "Validation",
        "ROC-AUC": roc_auc_score(y_valid, valid_pred),
        "Average Precision": average_precision_score(
            y_valid, valid_pred
        ),
        "LogLoss": log_loss(y_valid, valid_pred),
    },
    {
        "split": "Test",
        "ROC-AUC": roc_auc_score(y_test, test_pred),
        "Average Precision": average_precision_score(
            y_test, test_pred
        ),
        "LogLoss": log_loss(y_test, test_pred),
    },
])

display(classification_results.round(5))

classification_results.to_csv(
    OUTPUT_DIR / "classification_metrics.csv",
    index=False,
)

,split,ROC-AUC,Average Precision,LogLoss
0,Validation,0.93019,0.53573,0.06490
1,Test,0.93182,0.52441,0.06475


In [6]:
def ranking_metrics(df, score_col="score", k=10):
    ndcg_values = []
    recall_values = []
    precision_values = []

    for _, group in df.groupby(
        ["user_id", "cutoff_date"], sort=False
    ):
        group = group.sort_values(
            score_col, ascending=False
        )

        labels = group["label"].to_numpy()
        top_labels = labels[:k]

        discounts = 1 / np.log2(
            np.arange(2, len(top_labels) + 2)
        )

        dcg = np.sum(top_labels * discounts)

        ideal_labels = np.sort(labels)[::-1][:k]
        ideal_discounts = 1 / np.log2(
            np.arange(2, len(ideal_labels) + 2)
        )
        idcg = np.sum(ideal_labels * ideal_discounts)

        ndcg_values.append(
            dcg / idcg if idcg > 0 else 0.0
        )

        total_positives = labels.sum()

        recall_values.append(
            top_labels.sum() / total_positives
            if total_positives > 0 else 0.0
        )

        precision_values.append(
            top_labels.sum() / k
        )

    return {
        "NDCG@10": np.mean(ndcg_values),
        "Recall@10": np.mean(recall_values),
        "Precision@10": np.mean(precision_values),
        "groups": len(ndcg_values),
    }


valid_ranking = valid[
    ["user_id", "item_id", "cutoff_date", "label"]
].copy()

test_ranking = test[
    ["user_id", "item_id", "cutoff_date", "label"]
].copy()

valid_ranking["score"] = valid_pred
test_ranking["score"] = test_pred

catboost_valid_metrics = ranking_metrics(
    valid_ranking, k=TOP_K
)
catboost_test_metrics = ranking_metrics(
    test_ranking, k=TOP_K
)

print("CatBoost — Validation")
display(pd.DataFrame([catboost_valid_metrics]).round(5))

print("CatBoost — Test")
display(pd.DataFrame([catboost_test_metrics]).round(5))

CatBoost — Validation


,NDCG@10,Recall@10,Precision@10,groups
0,0.70538,0.80166,0.18576,1650


CatBoost — Test


,NDCG@10,Recall@10,Precision@10,groups
0,0.6991,0.8048,0.18428,3461


In [7]:
popularity_column = "item_interactions_30d"

if popularity_column not in valid.columns:
    raise KeyError(
        f"Нет признака {popularity_column}. "
        "Проверь названия колонок в parquet."
    )

valid_popularity = valid[
    ["user_id", "item_id", "cutoff_date", "label",
     popularity_column]
].copy()

test_popularity = test[
    ["user_id", "item_id", "cutoff_date", "label",
     popularity_column]
].copy()

valid_popularity["score"] = pd.to_numeric(
    valid_popularity[popularity_column], errors="coerce"
).fillna(0)

test_popularity["score"] = pd.to_numeric(
    test_popularity[popularity_column], errors="coerce"
).fillna(0)

popularity_valid_metrics = ranking_metrics(
    valid_popularity, k=TOP_K
)

popularity_test_metrics = ranking_metrics(
    test_popularity, k=TOP_K
)

print("Popularity — Validation")
display(pd.DataFrame([popularity_valid_metrics]).round(5))

print("Popularity — Test")
display(pd.DataFrame([popularity_test_metrics]).round(5))

Popularity — Validation


,NDCG@10,Recall@10,Precision@10,groups
0,0.68667,0.78737,0.18145,1650


Popularity — Test


,NDCG@10,Recall@10,Precision@10,groups
0,0.6742,0.78523,0.17819,3461


In [8]:
comparison = pd.DataFrame([
    {
        "model": "CatBoost — manual features",
        **catboost_test_metrics,
    },
    {
        "model": "Popularity",
        **popularity_test_metrics,
    },
])

comparison.to_csv(
    OUTPUT_DIR / "ranking_comparison.csv",
    index=False,
)

print("Сравнение на TEST:")
display(comparison.round(5))

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.get_feature_importance(),
}).sort_values("importance", ascending=False)

print("Топ-15 признаков:")
display(feature_importance.head(15))

feature_importance.to_csv(
    OUTPUT_DIR / "feature_importance.csv",
    index=False,
)

model.save_model(
    str(OUTPUT_DIR / "catboost_baseline.cbm")
)

print("Результаты сохранены в:", OUTPUT_DIR.resolve())

Сравнение на TEST:


,model,NDCG@10,Recall@10,Precision@10,groups
0,CatBoost — manual features,0.6991,0.80480,0.18428,3461
1,Popularity,0.6742,0.78523,0.17819,3461


Топ-15 признаков:


,feature,importance
1,item_id,16.427727
12,item_interactions_30d,11.059374
10,item_mean_watched_pct,8.487485
0,user_id,8.104849
8,item_history_count,7.511220
14,item_days_since_last_watch,6.884256
13,item_interactions_90d,6.813054
11,item_interactions_7d,5.720540
23,user_genre_preference,5.080001
16,release_year,4.512641


Результаты сохранены в: D:\tlab-research-1\results_baseline


CatBoost превзошел baseline популярности по всем метрикам ранжирования. Наибольший вклад в качество модели вносят item_id, частота взаимодействий с фильмом за последние 30 дней (item_interactions_30d), средний процент просмотра (item_mean_watched_pct) и user_id. Это показывает, что помимо популярности модель учитывает историю взаимодействий и особенности пользователей. Полученный baseline будет использоваться для сравнения с моделью, дополненной автоматически сгенерированными признаками.

## Эксперимент

## Исследовательская идея

В качестве исследовательского подхода использована автоматизация построения признаков на основе истории взаимодействий пользователей с контентом. Для пользователей рассчитываются агрегаты активности, давности просмотров и статистики потребления контента. Для фильмов рассчитываются частоты взаимодействий, число уникальных пользователей и статистики просмотров. Дополнительно учитываются временные окна в 7, 30 и 90 дней.

Автоматически сгенерированные признаки объединяются с ручными признаками и передаются в CatBoost. Такой подход позволяет проверить, улучшает ли автоматизация feature engineering качество рекомендаций по сравнению с вручную разработанным набором признаков.

## Экспериментальное сравнение

Сравниваются три подхода: ранжирование по популярности, CatBoost с ручными признаками и CatBoost с объединёнными ручными и автоматически сгенерированными признаками. Для моделей CatBoost проводится ограниченный поиск гиперпараметров по валидационной выборке. Основной критерий выбора — NDCG@10. Тестовая выборка используется только для итоговой оценки выбранной конфигурации.

Качество оценивается по NDCG@10, Recall@10 и Precision@10. Дополнительно рассчитываются ROC-AUC, Average Precision и LogLoss. Для оценки вычислительных затрат фиксируются время генерации признаков, время обучения и объём памяти, занимаемый подготовленными таблицами.

## Анализ результатов

Сравнение позволяет определить, даёт ли автоматическое построение признаков дополнительный сигнал по сравнению с ручным feature engineering. Если объединённая модель показывает более высокий NDCG@10 на тесте, автоматические признаки улучшают ранжирование. Если ROC-AUC и Average Precision растут, но NDCG@10 остаётся практически неизменным, улучшение классификации не приводит к заметному улучшению верхней части рекомендованного списка.

Важность признаков используется для определения агрегатов, которые вносят наибольший вклад в предсказания. Вычислительные затраты сопоставляются с изменением качества, чтобы оценить практическую целесообразность усложнения пайплайна.

## Ограничения

Отсутствие взаимодействия не означает отрицательное отношение пользователя к контенту: непросмотренные фильмы используются как неявные отрицательные примеры. Метрики ранжирования рассчитываются на сформированном наборе кандидатов и не полностью отражают качество рекомендаций на всём каталоге.

Результаты зависят от временного разбиения, состава кандидатов и выбранного набора пользователей. Поэтому небольшое изменение метрик не следует считать убедительным преимуществом без повторных временных экспериментов. Автоматическое построение признаков также увеличивает вычислительные затраты и требует строгого ограничения истории по времени для предотвращения утечки будущей информации.


In [9]:
from pathlib import Path
from time import perf_counter
import warnings
import gc

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
)

warnings.filterwarnings("ignore")

SEED = 42
TOP_K = 10

DATA_DIR = Path("data_processed_100neg")
if not DATA_DIR.exists():
    DATA_DIR = Path("data_processed")

ORIGINAL_DIR = Path("data_original")
OUTPUT_DIR = Path("results_feature_engineering")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)


def load_split(name):
    path = DATA_DIR / f"{name}_features.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

    df = pd.read_parquet(path)
    df["cutoff_date"] = pd.to_datetime(df["cutoff_date"])
    df["label"] = df["label"].astype(int)

    print(
        f"{name}: rows={len(df):,}, "
        f"users={df.user_id.nunique():,}, "
        f"items={df.item_id.nunique():,}, "
        f"positive rate={df.label.mean():.4f}"
    )
    return df


train = load_split("train")
valid = load_split("valid")
test = load_split("test")

for name, df in [("train", train), ("valid", valid), ("test", test)]:
    assert {"user_id", "item_id", "cutoff_date", "label"} <= set(df.columns)
    assert df["label"].isin([0, 1]).all()

print("\nДанные загружены.")


excluded = {"label", "cutoff_date"}

manual_features = [
    c for c in train.columns
    if c not in excluded
    and c in valid.columns
    and c in test.columns
]

categorical_manual = [
    c for c in manual_features
    if (
        c in {"user_id", "item_id"}
        or train[c].dtype == "object"
        or isinstance(train[c].dtype, pd.CategoricalDtype)
    )
]

print("Количество ручных признаков:", len(manual_features))
print("Категориальные признаки:", categorical_manual)

interaction_path = ORIGINAL_DIR / "interactions.csv"
if not interaction_path.exists():
    interaction_path = Path("interactions.csv")

if not interaction_path.exists():
    raise FileNotFoundError(
        "Не найден interactions.csv в data_original или текущей папке."
    )

history = pd.read_csv(interaction_path)
history["last_watch_dt"] = pd.to_datetime(
    history["last_watch_dt"], errors="coerce"
)

all_split_users = pd.Index(
    pd.concat([
        train["user_id"],
        valid["user_id"],
        test["user_id"],
    ]).dropna().unique()
)

# Оставляем историю только для пользователей эксперимента.
history = history[history["user_id"].isin(all_split_users)].copy()
history = history.dropna(
    subset=["user_id", "item_id", "last_watch_dt"]
)

for col in ["total_dur", "watched_pct"]:
    if col in history.columns:
        history[col] = pd.to_numeric(history[col], errors="coerce")

print("Исторических взаимодействий:", f"{len(history):,}")
print("Пользователей в истории:", f"{history.user_id.nunique():,}")


def generate_auto_features(df, history):
    start = perf_counter()
    parts = []

    numeric_columns = [
        c for c in ["total_dur", "watched_pct"]
        if c in history.columns
    ]

    for cutoff, part in df.groupby("cutoff_date", sort=False):
        cutoff = pd.Timestamp(cutoff)

        # Только информация, известная на момент рекомендации.
        hist = history[
            history["last_watch_dt"] <= cutoff
        ].copy()

        hist["days_ago"] = (
            cutoff.normalize()
            - hist["last_watch_dt"].dt.normalize()
        ).dt.days.clip(lower=0)

        # User aggregates
        user_agg = {
            "item_id": ["count", "nunique"],
            "days_ago": ["min", "mean", "max"],
        }
        for col in numeric_columns:
            user_agg[col] = ["mean", "std", "min", "max"]

        user_stats = hist.groupby("user_id").agg(user_agg)
        user_stats.columns = [
            "auto_user_" + "_".join(map(str, col))
            for col in user_stats.columns
        ]
        user_stats = user_stats.reset_index()

        # Item aggregates
        item_agg = {
            "user_id": ["count", "nunique"],
            "days_ago": ["min", "mean"],
        }
        for col in numeric_columns:
            item_agg[col] = ["mean", "std", "min", "max"]

        item_stats = hist.groupby("item_id").agg(item_agg)
        item_stats.columns = [
            "auto_item_" + "_".join(map(str, col))
            for col in item_stats.columns
        ]
        item_stats = item_stats.reset_index()

        # Activity in recent time windows
        result = part.copy()

        for days in [7, 30, 90]:
            recent = hist[hist["days_ago"] < days]

            user_counts = recent.groupby("user_id").size()
            item_counts = recent.groupby("item_id").size()

            result[f"auto_user_interactions_{days}d"] = (
                result["user_id"].map(user_counts).fillna(0)
            )
            result[f"auto_item_interactions_{days}d"] = (
                result["item_id"].map(item_counts).fillna(0)
            )

        result = result.merge(
            user_stats, on="user_id", how="left", sort=False
        )
        result = result.merge(
            item_stats, on="item_id", how="left", sort=False
        )

        parts.append(result)

    result = pd.concat(parts, ignore_index=True)

    # Missing aggregate means no history for this entity.
    auto_cols = [
        c for c in result.columns
        if c.startswith("auto_")
    ]
    for col in auto_cols:
        result[col] = pd.to_numeric(
            result[col], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan).fillna(0)

    print(
        f"Generated {len(auto_cols)} features for "
        f"{len(df):,} rows in {perf_counter() - start:.1f} sec."
    )
    return result, auto_cols


print("\nGenerating automatic features...")
feature_start = perf_counter()

train_auto, auto_features = generate_auto_features(train, history)
valid_auto, _ = generate_auto_features(valid, history)
test_auto, _ = generate_auto_features(test, history)

feature_generation_time = perf_counter() - feature_start

# Save generated features for reproducibility
train_auto.to_parquet(OUTPUT_DIR / "train_auto.parquet", index=False)
valid_auto.to_parquet(OUTPUT_DIR / "valid_auto.parquet", index=False)
test_auto.to_parquet(OUTPUT_DIR / "test_auto.parquet", index=False)

combined_features = list(dict.fromkeys(
    manual_features + auto_features
))

print("\nManual features:", len(manual_features))
print("Automatic features:", len(auto_features))
print("Combined features:", len(combined_features))
print(f"Feature generation total: {feature_generation_time:.1f} sec")

def ranking_metrics(df, score_col="score", k=10):
    ndcg_values = []
    recall_values = []
    precision_values = []

    for _, group in df.groupby(
        ["user_id", "cutoff_date"], sort=False
    ):
        group = group.sort_values(score_col, ascending=False)
        labels = group["label"].to_numpy()

        top_labels = labels[:k]
        discounts = 1 / np.log2(np.arange(2, len(top_labels) + 2))
        dcg = np.sum(top_labels * discounts)

        ideal = np.sort(labels)[::-1][:k]
        ideal_discounts = 1 / np.log2(np.arange(2, len(ideal) + 2))
        idcg = np.sum(ideal * ideal_discounts)

        ndcg_values.append(dcg / idcg if idcg > 0 else 0.0)

        total_positive = labels.sum()
        recall_values.append(
            top_labels.sum() / total_positive
            if total_positive else 0.0
        )
        precision_values.append(top_labels.sum() / k)

    return {
        "NDCG@10": float(np.mean(ndcg_values)),
        "Recall@10": float(np.mean(recall_values)),
        "Precision@10": float(np.mean(precision_values)),
        "groups": len(ndcg_values),
    }


def classification_metrics(y, pred):
    return {
        "ROC-AUC": roc_auc_score(y, pred),
        "Average Precision": average_precision_score(y, pred),
        "LogLoss": log_loss(y, pred),
    }


def prepare_features(df, columns, categorical_columns):
    X = df[columns].copy()

    for col in columns:
        if col in categorical_columns:
            X[col] = X[col].fillna("__MISSING__").astype(str)
        else:
            X[col] = pd.to_numeric(
                X[col], errors="coerce"
            ).replace([np.inf, -np.inf], np.nan).fillna(0)

    return X


parameter_grid = [
    {"depth": 6, "learning_rate": 0.05},
    {"depth": 8, "learning_rate": 0.05},
]

def fit_and_tune(name, feature_cols, train_df, valid_df, test_df):
    cats = [
        c for c in feature_cols
        if (
            c in {"user_id", "item_id"}
            or train_df[c].dtype == "object"
            or isinstance(train_df[c].dtype, pd.CategoricalDtype)
        )
    ]

    X_train = prepare_features(train_df, feature_cols, cats)
    X_valid = prepare_features(valid_df, feature_cols, cats)
    X_test = prepare_features(test_df, feature_cols, cats)

    candidates = []
    tuning_start = perf_counter()

    for params in parameter_grid:
        print(f"\n{name}: testing {params}")

        model = CatBoostClassifier(
            iterations=1200,
            depth=params["depth"],
            learning_rate=params["learning_rate"],
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=SEED,
            early_stopping_rounds=100,
            allow_writing_files=False,
            verbose=False,
            thread_count=-1,
        )

        fit_start = perf_counter()
        model.fit(
            X_train,
            train_df["label"],
            cat_features=cats,
            eval_set=(X_valid, valid_df["label"]),
            use_best_model=True,
            verbose=False,
        )
        fit_seconds = perf_counter() - fit_start

        valid_pred = model.predict_proba(X_valid)[:, 1]

        valid_frame = valid_df[
            ["user_id", "item_id", "cutoff_date", "label"]
        ].copy()
        valid_frame["score"] = valid_pred

        rank_valid = ranking_metrics(valid_frame, k=TOP_K)
        cls_valid = classification_metrics(
            valid_df["label"], valid_pred
        )

        candidates.append({
            "name": name,
            "params": params,
            "model": model,
            "valid_pred": valid_pred,
            "valid_rank": rank_valid,
            "valid_cls": cls_valid,
            "fit_seconds": fit_seconds,
            "valid_ndcg": rank_valid["NDCG@10"],
        })

        print(
            f"  valid NDCG@10={rank_valid['NDCG@10']:.5f}; "
            f"AUC={cls_valid['ROC-AUC']:.5f}; "
            f"fit={fit_seconds:.1f}s"
        )

   
    best = max(candidates, key=lambda x: x["valid_ndcg"])

    test_pred = best["model"].predict_proba(X_test)[:, 1]

    test_frame = test_df[
        ["user_id", "item_id", "cutoff_date", "label"]
    ].copy()
    test_frame["score"] = test_pred

    best["test_pred"] = test_pred
    best["test_rank"] = ranking_metrics(test_frame, k=TOP_K)
    best["test_cls"] = classification_metrics(
        test_df["label"], test_pred
    )
    best["tuning_seconds"] = perf_counter() - tuning_start
    best["feature_count"] = len(feature_cols)
    best["best_iteration"] = best["model"].best_iteration_

    return best


print("\nTuning CatBoost with manual features...")
manual_result = fit_and_tune(
    "CatBoost — Manual",
    manual_features, train, valid, test
)

print("\nTuning CatBoost with manual + automatic features...")
combined_result = fit_and_tune(
    "CatBoost — Manual + Auto",
    combined_features, train_auto, valid_auto, test_auto
)


popularity_column = "item_interactions_30d"

if popularity_column not in valid.columns:
    raise KeyError(
        f"Popularity baseline needs {popularity_column} in prepared features."
    )

def score_popularity(df):
    out = df[
        ["user_id", "item_id", "cutoff_date", "label"]
    ].copy()
    out["score"] = pd.to_numeric(
        df[popularity_column], errors="coerce"
    ).fillna(0)
    return out

pop_valid = score_popularity(valid)
pop_test = score_popularity(test)

popularity_valid_rank = ranking_metrics(pop_valid, k=TOP_K)
popularity_test_rank = ranking_metrics(pop_test, k=TOP_K)

rows = []

for result in [manual_result, combined_result]:
    rows.append({
        "model": result["name"],
        "features": result["feature_count"],
        "best_depth": result["params"]["depth"],
        "best_learning_rate": result["params"]["learning_rate"],
        "best_iteration": result["best_iteration"],
        "Validation ROC-AUC": result["valid_cls"]["ROC-AUC"],
        "Validation AP": result["valid_cls"]["Average Precision"],
        "Validation NDCG@10": result["valid_rank"]["NDCG@10"],
        "Test ROC-AUC": result["test_cls"]["ROC-AUC"],
        "Test AP": result["test_cls"]["Average Precision"],
        "Test LogLoss": result["test_cls"]["LogLoss"],
        "Test NDCG@10": result["test_rank"]["NDCG@10"],
        "Test Recall@10": result["test_rank"]["Recall@10"],
        "Test Precision@10": result["test_rank"]["Precision@10"],
        "Fit time (sec)": result["tuning_seconds"],
    })

rows.append({
    "model": "Popularity",
    "features": 1,
    "best_depth": np.nan,
    "best_learning_rate": np.nan,
    "best_iteration": np.nan,
    "Validation ROC-AUC": np.nan,
    "Validation AP": np.nan,
    "Validation NDCG@10": popularity_valid_rank["NDCG@10"],
    "Test ROC-AUC": np.nan,
    "Test AP": np.nan,
    "Test LogLoss": np.nan,
    "Test NDCG@10": popularity_test_rank["NDCG@10"],
    "Test Recall@10": popularity_test_rank["Recall@10"],
    "Test Precision@10": popularity_test_rank["Precision@10"],
    "Fit time (sec)": 0.0,
})

comparison = pd.DataFrame(rows)
comparison.to_csv(
    OUTPUT_DIR / "model_comparison.csv", index=False
)

print("\n========== MODEL COMPARISON ==========")
display(comparison.round(5))

importance = pd.DataFrame({
    "feature": combined_features,
    "importance": combined_result["model"].get_feature_importance(),
}).sort_values("importance", ascending=False)

importance.to_csv(
    OUTPUT_DIR / "combined_feature_importance.csv", index=False
)

print("\n========== TOP 20 FEATURES ==========")
display(importance.head(20))

print("\n========== AUTOMATIC FEATURES IMPORTANCE ==========")
display(
    importance[importance["feature"].str.startswith("auto_")]
    .head(20)
)

def dataframe_size_mb(df):
    return df.memory_usage(index=True, deep=True).sum() / (1024 ** 2)

resource_summary = pd.DataFrame([
    {
        "stage": "Feature generation",
        "seconds": feature_generation_time,
        "input_rows": len(train) + len(valid) + len(test),
        "feature_count": len(auto_features),
        "dataframe_size_mb": (
            dataframe_size_mb(train_auto)
            + dataframe_size_mb(valid_auto)
            + dataframe_size_mb(test_auto)
        ),
    },
    {
        "stage": "CatBoost Manual tuning",
        "seconds": manual_result["tuning_seconds"],
        "input_rows": len(train) + len(valid) + len(test),
        "feature_count": len(manual_features),
        "dataframe_size_mb": (
            dataframe_size_mb(train)
            + dataframe_size_mb(valid)
            + dataframe_size_mb(test)
        ),
    },
    {
        "stage": "CatBoost Manual + Auto tuning",
        "seconds": combined_result["tuning_seconds"],
        "input_rows": len(train_auto) + len(valid_auto) + len(test_auto),
        "feature_count": len(combined_features),
        "dataframe_size_mb": (
            dataframe_size_mb(train_auto)
            + dataframe_size_mb(valid_auto)
            + dataframe_size_mb(test_auto)
        ),
    },
])

resource_summary.to_csv(
    OUTPUT_DIR / "resource_summary.csv", index=False
)

print("\n========== COMPUTE COSTS ==========")
display(resource_summary.round(2))

# Save selected models
manual_result["model"].save_model(
    str(OUTPUT_DIR / "catboost_manual.cbm")
)
combined_result["model"].save_model(
    str(OUTPUT_DIR / "catboost_manual_auto.cbm")
)

print("\nSaved files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

print("\nExperiment completed.")

train: rows=1,799,113, users=7,772, items=15,963, positive rate=0.0226
valid: rows=169,207, users=1,650, items=15,962, positive rate=0.0249
test: rows=354,870, users=2,777, items=15,963, positive rate=0.0247

Данные загружены.
Количество ручных признаков: 25
Категориальные признаки: ['user_id', 'item_id']
Исторических взаимодействий: 56,875
Пользователей в истории: 9,493

Generating automatic features...
Generated 31 features for 1,799,113 rows in 1.3 sec.
Generated 31 features for 169,207 rows in 0.1 sec.
Generated 31 features for 354,870 rows in 0.2 sec.

Manual features: 25
Automatic features: 31
Combined features: 56
Feature generation total: 1.6 sec

Tuning CatBoost with manual features...

CatBoost — Manual: testing {'depth': 6, 'learning_rate': 0.05}
  valid NDCG@10=0.70579; AUC=0.93060; fit=233.1s

CatBoost — Manual: testing {'depth': 8, 'learning_rate': 0.05}
  valid NDCG@10=0.70538; AUC=0.93019; fit=192.6s

Tuning CatBoost with manual + automatic features...

CatBoost — Manua

,model,features,best_depth,best_learning_rate,best_iteration,Validation ROC-AUC,Validation AP,Validation NDCG@10,Test ROC-AUC,Test AP,Test LogLoss,Test NDCG@10,Test Recall@10,Test Precision@10,Fit time (sec)
0,CatBoost — Manual,25,6.0,0.05,509.0,0.93060,0.53337,0.70579,0.93315,0.52102,0.06496,0.70153,0.80516,0.18451,428.32224
1,CatBoost — Manual + Auto,56,6.0,0.05,413.0,0.92973,0.53677,0.70657,0.93143,0.52483,0.06514,0.69945,0.80231,0.18370,407.42030
2,Popularity,1,NaN,NaN,NaN,NaN,NaN,0.68667,NaN,NaN,NaN,0.67420,0.78523,0.17819,0.00000



========== TOP 20 FEATURES ==========


,feature,importance
1,item_id,20.144908
47,auto_item_days_ago_mean,13.285736
0,user_id,8.292418
23,user_genre_preference,7.721326
55,auto_item_watched_pct_max,7.406786
12,item_interactions_30d,4.803141
11,item_interactions_7d,2.721472
22,genre_overlap_count,2.709958
46,auto_item_days_ago_min,2.256475
30,auto_item_interactions_90d,2.068149



========== AUTOMATIC FEATURES IMPORTANCE ==========


,feature,importance
47,auto_item_days_ago_mean,13.285736
55,auto_item_watched_pct_max,7.406786
46,auto_item_days_ago_min,2.256475
30,auto_item_interactions_90d,2.068149
34,auto_user_days_ago_mean,1.888912
53,auto_item_watched_pct_std,1.874159
52,auto_item_watched_pct_mean,1.834824
26,auto_item_interactions_7d,1.493383
48,auto_item_total_dur_mean,1.428341
27,auto_user_interactions_30d,1.422532



========== COMPUTE COSTS ==========


,stage,seconds,input_rows,feature_count,dataframe_size_mb
0,Feature generation,1.65,2323190,31,1192.94
1,CatBoost Manual tuning,428.32,2323190,25,643.48
2,CatBoost Manual + Auto tuning,407.42,2323190,56,1192.94



Saved files:
 - catboost_manual.cbm
 - catboost_manual_auto.cbm
 - combined_feature_importance.csv
 - model_comparison.csv
 - resource_summary.csv
 - test_auto.parquet
 - train_auto.parquet
 - valid_auto.parquet

Experiment completed.


## Итог эксперимента
Автоматическое построение признаков позволило увеличить Average Precision на тестовой выборке с 0.52102 до 0.52483, однако не привело к улучшению ранжирования: NDCG@10 снизился с 0.70153 до 0.69945, а Recall@10 — с 0.80516 до 0.80231. Обе модели CatBoost превосходят baseline популярности по NDCG@10.

Среди автоматически сгенерированных признаков наибольшую важность имеют средняя давность взаимодействий с фильмом  и максимальный процент просмотра. Это показывает, что агрегаты истории содержат полезную информацию, но не обеспечивают прироста качества верхней части рекомендательного списка.

## Вывод
Автоматизация улучшила метрику Average Precision, но не повысила качество ранжирования по NDCG@10 и Recall@10. Поэтому в текущем эксперименте преимущество автоматического построения признаков неоднозначно. Для рекомендательной системы целесообразно ориентироваться прежде всего на метрики ранжирования и оценивать, оправдывает ли дополнительный объем данных полученный результат.

### Можно ли сделать так, чтобы построение признаков не ограничивалось решениями человека?

Да, построение признаков можно автоматизировать и сделать менее зависимым от решений человека. В эксперименте для рекомендательной системы были автоматически сформированы 31 дополнительный признак на основе истории взаимодействий пользователей с фильмами: частота просмотров, давность взаимодействий и статистики времени просмотра.

Однако эксперимент показал, что автоматическое построение признаков не гарантирует улучшения качества модели. На тестовой выборке Average Precision выросла с 0.52102 до 0.52483, но NDCG@10 снизилась с 0.70153 до 0.69945, а Recall@10 — с 0.80516 до 0.80231.

Таким образом, автоматизация построения признаков возможна, но полностью исключать человека нецелесообразно.